# Геокодирование городов из CRM-данных
Этот ноутбук предназначен для получения географических координат (широта и долгота) для городов, найденных в данных о сделках. Он использует библиотеку `geopy` и сервис Nominatim (OpenStreetMap).

In [1]:
import os
import time
import json
import pandas as pd
from geopy.geocoders import Nominatim

# Пути к файлам (относительно папки notebooks/)
CLEANED_DIR = os.path.join('..', 'data', 'cleaned')
OUT_FILE    = os.path.join('..', 'data', 'city_coords.json')

# Список стран для уточнения поиска
COUNTRY_FALLBACKS = ['Germany', 'Ukraine', 'Poland', 'Austria', 'Switzerland', '']

In [2]:
def geocode_city(geolocator, city):
    """Пробует несколько вариантов запроса, возвращает [lat, lon] или None."""
    for country in COUNTRY_FALLBACKS:
        query = f'{city}, {country}' if country else city
        try:
            loc = geolocator.geocode(query, exactly_one=True)
            if loc:
                time.sleep(1.2)   # соблюдение лимитов Nominatim (1 зап/сек)
                return [loc.latitude, loc.longitude]
        except Exception as e:
            if "429" in str(e):
                print(f'    Ошибка 429 (Too Many Requests) для {query}. Пауза 5 сек...')
                time.sleep(5.0)
            else:
                print(f'    Ошибка при запросе ({query}): {e}')
                time.sleep(2.0)
    return None

In [3]:
# Загружаем сделки для получения списка городов
deals_path = os.path.join(CLEANED_DIR, 'deals_clean.pkl')
deals = pd.read_pickle(deals_path)

# Исключаем системные значения
city_counts = (
    deals[~deals['city'].isin(['Unknown', '-', '', 'None'])]
    .groupby('city', observed=True)['id']
    .count()
    .sort_values(ascending=False)
)
all_cities = city_counts.index.tolist()
print(f'Всего уникальных городов в данных: {len(all_cities)}')

# Загружаем существующий кэш
if os.path.exists(OUT_FILE):
    with open(OUT_FILE, 'r', encoding='utf-8') as f:
        city_coords = json.load(f)
    print(f'Загружено из кэша: {len(city_coords)} городов')
else:
    city_coords = {}
    print('Кэш отсутствует, начинаем с нуля.')

# Список городов для геокодирования
to_geocode = [c for c in all_cities if c not in city_coords]
print(f'Нужно геокодировать: {len(to_geocode)} городов')

Всего уникальных городов в данных: 871
Загружено из кэша: 554 городов
Нужно геокодировать: 318 городов


In [4]:
geolocator = Nominatim(user_agent='crm_geo_analysis_ipynb_v1', timeout=10)
not_found = []

# Запуск геокодирования
for i, city in enumerate(to_geocode, 1):
    coords = geocode_city(geolocator, city)
    if coords:
        city_coords[city] = coords
        print(f'  [{i}/{len(to_geocode)}] {city}: {coords[0]:.4f}, {coords[1]:.4f}')
    else:
        not_found.append(city)
        print(f'  [{i}/{len(to_geocode)}] {city}: НЕ НАЙДЕН')

    # Сохраняем промежуточные результаты каждые 10 городов
    if i % 10 == 0:
        with open(OUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(city_coords, f, ensure_ascii=False, indent=2)

# Финальное сохранение
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(city_coords, f, ensure_ascii=False, indent=2)

print("\nГеокодирование завершено.")
print(f"Итого в кэше: {len(city_coords)} городов.")
if not_found:
    print(f"Не найдены координаты для: {not_found}")

  [1/318] Annaberg-Buchholz: 50.5789, 13.0106
  [2/318] Postbauer-Heng: 49.3030, 11.3507
  [3/318] Wetzlar: 50.5706, 8.5312
  [4/318] Trebgast: 50.0680, 11.5515
  [5/318] Porta Westfalica: 52.2394, 8.9248
  [6/318] Poppenhausen: 50.0996, 10.1438
  [7/318] Wiefelstede: 53.2559, 8.1150
  [8/318] Trossingen: 48.0751, 8.6363
  [9/318] Poing: 48.1667, 11.8037
  [10/318] Podskalie: 49.0444, 18.4544
  [11/318] Plön: 54.1581, 10.4177
  [12/318] Plauen: 50.4951, 12.1347
  [13/318] Planegg: 48.1037, 11.4220
  [14/318] Piotrków Trybunalski: 51.4129, 19.6887
  [15/318] Wiesenttal: 49.8263, 11.2397
  [16/318] Wilhelmshaven: 53.5279, 8.1063
  [17/318] Pfinztal: 48.9919, 8.5553
  [18/318] Adelebsen: 51.5795, 9.7524
  [19/318] Pinneberg: 53.7279, 9.6980
  [20/318] Piatek: 49.3834, 8.2928
  [21/318] Tettnang: 47.6717, 9.5891
  [22/318] Phuket: 49.4131, 8.7097
  [23/318] Pleinfeld: 49.1054, 10.9856
  [24/318] Rees: 51.7581, 6.3957
  [25/318] Recklinghausen: 51.6144, 7.1979
  [26/318] Rechitsa: 51.6955, 